# Comment Classification: LLM Selection & Evaluation

Before running this notebook:
1. Install Ollama: https://ollama.com
2. To pull the models, run in terminal: 
    - ollama pull qwen3:8b
    - ollama pull llama3.1:8b

In [1]:
# importing libraries

import pandas as pd
import numpy as np
import ollama
import json
import re
from tqdm import tqdm
import krippendorff
from sklearn.metrics import classification_report, cohen_kappa_score

In [17]:
# loading the complete comments dataset
data_cpath = "data/comments/comments.csv"
data_comments = pd.read_csv(data_cpath, encoding="latin1")

# loading manually coded dataset for evaluation
data_epath = "data/comments/comments_eval.xlsx"
data_eval = pd.read_excel(data_epath, engine="openpyxl")
PROMPT_IDS = [2, 3, 6, 20, 27, 28, 31, 32, 34, 57, 98, 99, 104, 111, 129]
data_eval = data_eval[~data_eval["unique_id"].isin(PROMPT_IDS)].copy()
print(f"{len(data_eval)} Zeilen im Testset")


135 Zeilen im Testset


Here, we collapse the `breadth` variable.  

In [18]:
data_eval["breadth"] = data_eval["breadth"].clip(upper=3)

In [19]:
# defining the system prompt inline with the codebook

with open("data/comments/system_prompt.txt", "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()

In [20]:
# sanity check: kommt der Systemprompt beim Modell an?
# sanity check
print(len(SYSTEM_PROMPT))
print("LENGTH IS NOT EVIDENCE" in SYSTEM_PROMPT)
print(SYSTEM_PROMPT[:300])


15887
True
You are coding German-language user comments for a social science study titled "Likes or
Dislikes, Gratifications or Concerns?", about a German 9-point anti-terrorism plan
("9-Punkte-Plan") / terrorism policy. The unit of analysis is a single user comment. Code the
comment as a whole on the six vari


In [21]:
# define classifier function with qwen3:8b set as default

from ollama import Client
client = Client(timeout=120)
def classify_comment(text, parent_text=None, model="qwen3:8b"):
    """Classify a single comment on all 6 codebook variables in one call.
    Returns a dict of 6 ints, or a dict of -1s if parsing fails."""
    fallback = {"pers_exp": -1, "emot_exp": -1, "pol_opin": -1,
                "breadth": -1, "valence": -1, "contr": -1}

    if not isinstance(text, str) or text.strip() == "":
        return {"pers_exp": 0, "emot_exp": 0, "pol_opin": 0,
                "breadth": 0, "valence": 4, "contr": 0}

    user_msg = f"Comment: {text}"
    if parent_text:
        user_msg += f"\n\n(This is a reply to the following parent comment: {parent_text})"

    try:
        response = client.chat(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_msg}
            ],
            format="json",
            think=False,
            keep_alive="30m",
            options={"temperature": 0, "num_ctx": 8192, "num_predict": 400}
        )
    except Exception as e:
        print(f"Fehler: {e}")
        return fallback

    out = response["message"]["content"].strip()

    match = re.search(r"\{.*\}", out, re.DOTALL)
    if not match:
        return fallback

    try:
        parsed = json.loads(match.group(0))
    except json.JSONDecodeError:
        return fallback

    result = {}
    bounds = {"pers_exp": (0,1), "emot_exp": (0,1), "pol_opin": (0,1),
              "breadth": (0,3), "valence": (1,4), "contr": (0,1)}
    for key, (lo, hi) in bounds.items():
        val = parsed.get(key, -1)
        try:
            val = int(val)
        except (TypeError, ValueError):
            val = -1
        result[key] = val if lo <= val <= hi else -1

    return result

In [22]:
# post_number resets per topic, so it's not unique on its own - build a composite key instead
def make_id(topic, post_number):
    return f"{topic}_{int(post_number)}"

# build ids on the FULL data_comments first, so parent lookups still work even for
# replies whose parent falls outside whatever subset we classify below
data_comments["row_id"] = data_comments.apply(lambda r: make_id(r["topic"], r["post_number"]), axis=1)
data_eval["row_id"] = data_eval.apply(lambda r: make_id(r["topic"], r["post_number"]), axis=1)
id_col = "row_id"

reply_col = "reply_to_post_number" if "reply_to_post_number" in data_comments.columns else None
text_by_id = dict(zip(data_comments["row_id"], data_comments["raw"].fillna("")))

def get_parent_text(row):
    if reply_col and pd.notna(row.get(reply_col)):
        parent_id = make_id(row["topic"], row[reply_col])
        return text_by_id.get(parent_id)
    return None

data_comments_full = data_comments.copy()  # keep the unfiltered version for the full run later

# for now, only classify comments that already have a manual code (validation pass) -
# comment this line out later to run the full dataset instead
data_comments = data_comments[data_comments[id_col].isin(data_eval[id_col])].copy()
print(f"Classifying {len(data_comments)} comments")

Classifying 135 comments


In [ ]:
MODELS = ["MichelRosselli/apertus:8b-instruct-2509-q4_k_m"]

In [24]:
VARS = ["pers_exp", "emot_exp", "pol_opin", "breadth", "valence", "contr"]
texts = data_comments["raw"].fillna("")


# delete:
#data_comments = data_comments.head().copy()


for model_name in MODELS:
    results = []
    for i, row in tqdm(data_comments.iterrows(), total=len(data_comments), desc=f"Classifying ({model_name})"):
        parent_text = get_parent_text(row)
        codes = classify_comment(texts.loc[i], parent_text=parent_text, model=model_name)
        results.append(codes)

    results_df = pd.DataFrame(results)
    model_tag = model_name.replace(":", "_").replace(".", "_")
    pred_cols = [f"{v}_{model_tag}" for v in VARS]
    data_comments[pred_cols] = results_df.values

Classifying (qwen3:8b): 100%|██████████| 135/135 [06:50<00:00,  3.04s/it]


In [25]:
r = client.chat(
    model="qwen3:8b",
    messages=[{"role": "system", "content": SYSTEM_PROMPT},
              {"role": "user", "content": "Comment: Heisse Luft, wie Alles, was von Merkel kommt."}],
    format="json",
    think=False,
    options={"temperature": 0, "num_ctx": 8192, "num_predict": 400}
)
print(repr(r["message"]["content"]))

'{"pers_exp": 0, "emot_exp": 0, "pol_opin": 1, "breadth": 1, "valence": 2, "contr": 1}'


In [26]:
for tag in ["qwen3_8b"]:
    cols = [f"{v}_{tag}" for v in VARS]
    print(tag, (data_comments[cols] == -1).sum().sum())
print(data_comments[[f"{v}_qwen3_8b" for v in VARS]].head(10))

qwen3_8b 1
     pers_exp_qwen3_8b  emot_exp_qwen3_8b  pol_opin_qwen3_8b  \
12                   0                  0                  1   
29                   0                  1                  1   
49                   0                  0                  1   
57                   0                  0                  1   
61                   0                  0                  1   
66                   1                  1                  1   
69                   0                  0                  1   
71                   0                  1                  1   
79                   0                  0                  1   
100                  0                  0                  1   

     breadth_qwen3_8b  valence_qwen3_8b  contr_qwen3_8b  
12                  2                 3               1  
29                  2                 3               1  
49                  3                 2               1  
57                  3                 1             

In [27]:
data_comments.head()

for tag in ["qwen3_8b", "llama3_1_8b"]:
    cols = [f"{v}_{tag}" for v in VARS]
    if all(c in data_comments.columns for c in cols):
        print(tag, (data_comments[cols] == -1).sum().sum())

qwen3_8b 1


In [28]:
# create report

VARS = ["pers_exp", "emot_exp", "pol_opin", "breadth", "valence", "contr"]
var_weights = {"pers_exp": None, "emot_exp": None, "pol_opin": None, "contr": None,
               "breadth": "linear", "valence": None}
var_levels = {"pers_exp": "nominal", "emot_exp": "nominal", "pol_opin": "nominal",
              "contr": "nominal", "breadth": "ordinal", "valence": "nominal"}

def krippendorff_alpha(y_true, y_pred, level):
    try:
        data = np.array([y_true, y_pred], dtype=float)
        return krippendorff.alpha(reliability_data=data, level_of_measurement=level)
    except (ZeroDivisionError, ValueError):
        return float("nan")

data_eval_renamed = data_eval.rename(columns={v: f"{v}_true" for v in VARS})
merged = data_eval_renamed.merge(data_comments, on=id_col)
print(f"Matched {len(merged)} of {len(data_eval)} rows\n")

all_rows = []

for model_name in MODELS:
    model_tag = model_name.replace(":", "_").replace(".", "_")
    print(f"\n########## {model_name} ##########")
    for var in VARS:
        y_true = merged[f"{var}_true"]
        y_pred = merged[f"{var}_{model_tag}"]

        mask = y_true.notna() & y_pred.notna() & (y_pred != -1)
        y_true_m, y_pred_m = y_true[mask], y_pred[mask]

        print(f"=== {var} (n={mask.sum()}) ===")
        report_dict = classification_report(y_true_m, y_pred_m, zero_division=0, output_dict=True)
        print(classification_report(y_true_m, y_pred_m, zero_division=0))

        kappa = cohen_kappa_score(y_true_m, y_pred_m, weights=var_weights[var])
        alpha = krippendorff_alpha(y_true_m.values, y_pred_m.values, var_levels[var])
        print(f"Cohen's Kappa: {kappa:.3f}   Krippendorff's alpha ({var_levels[var]}): {alpha:.3f}\n")

        for class_label, metrics in report_dict.items():
            if isinstance(metrics, dict):  # skips "accuracy", which is a bare float
                all_rows.append({
                    "model": model_name, "variable": var, "n": mask.sum(),
                    "class": class_label,
                    "precision": metrics["precision"], "recall": metrics["recall"],
                    "f1_score": metrics["f1-score"], "support": metrics["support"],
                    "accuracy": round(report_dict["accuracy"], 3),
                    "cohens_kappa": round(kappa, 3),
                    "krippendorff_alpha": round(alpha, 3) if not np.isnan(alpha) else "n/a",
                })

report_df = pd.DataFrame(all_rows)
report_df.to_excel("data/comments/eval_report_coding_v5.xlsx", index=False)
print("Saved eval_report_coding_v5.xlsx")

Matched 135 of 135 rows


########## qwen3:8b ##########
=== pers_exp (n=135) ===
              precision    recall  f1-score   support

           0       0.95      0.98      0.97       119
           1       0.83      0.62      0.71        16

    accuracy                           0.94       135
   macro avg       0.89      0.80      0.84       135
weighted avg       0.94      0.94      0.94       135

Cohen's Kappa: 0.682   Krippendorff's alpha (nominal): 0.682

=== emot_exp (n=134) ===
              precision    recall  f1-score   support

           0       0.84      0.97      0.90        99
           1       0.85      0.49      0.62        35

    accuracy                           0.84       134
   macro avg       0.85      0.73      0.76       134
weighted avg       0.84      0.84      0.83       134

Cohen's Kappa: 0.529   Krippendorff's alpha (nominal): 0.521

=== pol_opin (n=135) ===
              precision    recall  f1-score   support

           0       0.88      0.86  

In [29]:
merged.to_excel("data/comments/errors_qwen_v5.xlsx", index=False)
print("gespeichert")

gespeichert
